<a href="https://colab.research.google.com/github/TBDA-Classes/2026G3/blob/main/TBDA_F2_2_Secondary_Loads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# F2.2 – Secondary loads: characterisation and forecast where data allow (Teams D + B)

Goal: tell which secondary loads (Emerg. Lighting, Hand Dryer, RACK, Projector) are
predictable, which are constant, and which are unreliable.

Part 1 — Characterisation + data-quality ratings: Team B
Part 2 — Forecast and final verdicts: Team D (section marked TODO below)


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cedint_f1_hourly_clean.csv.gz")
df["timestamp_local"] = pd.to_datetime(df["timestamp_local"])
print(df["load_type"].unique())

/tmp/ipykernel_3603/3574993079.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cedint_f1_hourly_clean.csv.gz")


['HVAC' 'Plugs & Sockets' 'Emerg. Lighting' 'Hand Dryer' 'Lighting' 'RACK'
 'Projector']


In [2]:
secondary = ["Emerg. Lighting", "Hand Dryer", "RACK", "Projector"]
flags = ["flag_counter_reset", "flag_counter_spike", "flag_neg_large",
         "flag_low_coverage", "flag_power_spike"]

rows = []
for load in secondary:
    d = df[df["load_type"] == load]
    for zone in d["zone"].unique():
        s = d[(d["zone"] == zone) &
              (df["timestamp_local"].dt.year.isin([2021, 2022])) &
              (d["usable_for_model"] == 1)]
        if len(s) < 100:   # dead or tiny series
            rows.append({"load": load, "zone": zone, "hours": len(s),
                         "status": "EXCLUDED (not enough usable hours)"})
            continue
        e = s["energy_wh_model"]
        cv = e.std() / e.mean() if e.mean() > 0 else np.nan
        rows.append({
            "load": load, "zone": zone, "hours": len(s),
            "n_devices": s["n_devices"].iloc[0],
            "mean_wh": round(e.mean(), 1),
            "cv": round(cv, 2),
            "share_zero": round((e == 0).mean(), 3),
            "weekly_autocorr": round(e.autocorr(lag=168), 2),
            "n_flag_hours": int(s[flags].sum().sum()),
        })

scorecard = pd.DataFrame(rows)
print(scorecard.to_string(index=False))

/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &
/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &
/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &
/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &
/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &
/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &
/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &


           load zone  hours  n_devices  mean_wh    cv  share_zero  weekly_autocorr  n_flag_hours                             status
Emerg. Lighting   NW  17520       24.0     12.8  0.35       0.033             0.77          19.0                                NaN
Emerg. Lighting   SE  17520       14.0      1.7  0.28       0.006             0.77         173.0                                NaN
Emerg. Lighting   SW  17520       18.0     11.0  0.15       0.000             0.24         995.0                                NaN
     Hand Dryer   NW  17520        3.0      3.1 20.21       0.729             0.01          96.0                                NaN
     Hand Dryer   SW  17520        2.0     16.4  9.13       0.030             0.29         517.0                                NaN
           RACK   NW  17520        1.0    162.5  0.19       0.000             0.96           8.0                                NaN
           RACK   SW   8083        1.0    280.9  0.03       0.000           

/tmp/ipykernel_3603/705058196.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s = d[(d["zone"] == zone) &


In [3]:
# Known data issue (Team B): SW RACK counter stops in Dec 2021
rack_sw = df[(df["load_type"] == "RACK") & (df["zone"] == "SW") &
             (df["usable_for_model"] == 1)].sort_values("timestamp_local")
print("SW RACK last usable hour:", rack_sw["timestamp_local"].max())

SW RACK last usable hour: 2021-12-03 18:00:00


## Part 1 — Characterisation verdicts (Team B)

Scorecard evidence (2021–2022, usable hours only):

CONSTANT / PREDICTABLE loads:
– RACK NW: mean 162.5 Wh/h, CV = 0.19, weekly autocorr = 0.96 → highly regular.
– RACK SW: mean 280.9 Wh/h, regular too — BUT series stops 2021-12-03 18:00
  (counter defect). Forecast restricted to the covered period.
– Emerg. Lighting SW: CV = 0.15 (the most constant of all), share_zero = 0%.
– Emerg. Lighting NW: CV = 0.35, autocorr = 0.77 — regular; sign-corrected series.
– Emerg. Lighting SE: CV = 0.28, share_zero = 0.6%.

INTERMITTENT / NOT PREDICTABLE loads:
– Hand Dryer NW: CV = 20.2, 73% of hours at zero, weekly autocorr = 0.01
  → no memory, no pattern: not predictable. Documented as such, no model forced.
– Hand Dryer SW: CV = 9.1, 3% zeros but bursty — not predictable either.

EXCLUDED:
– Projector SE: dead series (0% usable) — not in the scorecard (no usable hours).

Data-quality ratings: see secondary_loads_ratings.md. Flag hours noted: Emerg. SW 995,
Hand Dryer SW 517 — mostly low-coverage periods, already excluded via usable_for_model.